In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# 建立假數據
random.seed(42)

tenors = ['1D', '1W', '1M', '3M', '6M', '12M']
makers = ['Alice', 'Bob', 'Charlie']
checkers = ['David', 'Eve', 'Alice']  # Alice同時是maker，製造SoD違規

dates = pd.date_range('2024-01-01', '2024-01-31')

rows = []
for date in dates:
    for tenor in tenors:
        # 正常報送
        rows.append({
            'submission_date': date,
            'tenor': tenor,
            'rate': round(random.uniform(1.5, 4.5), 4),
            'maker_id': random.choice(makers),
            'checker_id': random.choice(checkers),
            'submission_time': f"{random.randint(9,11)}:{random.randint(0,59):02d}"
        })

# 故意加入異常
# 1. 重複報送
rows.append({'submission_date': pd.Timestamp('2024-01-05'), 'tenor': '1M', 'rate': 2.5, 'maker_id': 'Bob', 'checker_id': 'David', 'submission_time': '10:15'})

# 2. SoD違規
rows.append({'submission_date': pd.Timestamp('2024-01-10'), 'tenor': '3M', 'rate': 3.1, 'maker_id': 'Alice', 'checker_id': 'Alice', 'submission_time': '09:45'})

df = pd.DataFrame(rows)
print(f"資料筆數：{len(df)}")
df.head(10)


資料筆數：188


,submission_date,tenor,rate,maker_id,checker_id,submission_time
0,2024-01-01,1D,3.4183,Alice,Alice,10:15
1,2024-01-01,1W,2.1696,Charlie,David,11:47
2,2024-01-01,1M,4.1765,Alice,Alice,10:02
3,2024-01-01,3M,1.5894,Alice,David,11:38
4,2024-01-01,6M,1.5796,Alice,Alice,11:44
5,2024-01-01,12M,3.1348,Alice,Eve,11:17
6,2024-01-02,1D,3.9283,Alice,David,11:27
7,2024-01-02,1W,2.5208,Alice,David,10:06
8,2024-01-02,1M,1.7782,Alice,Eve,10:38
9,2024-01-02,3M,2.2936,Alice,Alice,10:34


In [2]:
# 轉換時間格式
df['submission_time'] = pd.to_datetime(df['submission_time'], format='%H:%M').dt.time
deadline_internal = datetime.strptime('10:30', '%H:%M').time()
deadline_regulatory = datetime.strptime('11:00', '%H:%M').time()

# ① 重複報送
duplicates = df[df.duplicated(subset=['submission_date', 'tenor'], keep=False)]

# ② SoD違規
sod_violations = df[df['maker_id'] == df['checker_id']]

# ③ 超過deadline
late_internal = df[df['submission_time'] > deadline_internal]
late_regulatory = df[df['submission_time'] > deadline_regulatory]

# ④ 利率波動異常（前一天同天期±10%）
df_sorted = df.sort_values(['tenor', 'submission_date'])
df_sorted['prev_rate'] = df_sorted.groupby('tenor')['rate'].shift(1)
df_sorted['rate_change_pct'] = abs(df_sorted['rate'] - df_sorted['prev_rate']) / df_sorted['prev_rate'] * 100
rate_anomalies = df_sorted[df_sorted['rate_change_pct'] > 10]

# ⑤ 漏報（某天某天期沒有報送）
all_combinations = pd.MultiIndex.from_product([dates, tenors], names=['submission_date', 'tenor'])
actual_combinations = pd.MultiIndex.from_frame(df[['submission_date', 'tenor']].drop_duplicates())
missing = all_combinations.difference(actual_combinations)
missing_df = pd.DataFrame(missing.tolist(), columns=['submission_date', 'tenor'])

# 結果摘要
print("="*50)
print(f"① 重複報送：{len(duplicates)} 筆")
print(f"② SoD違規：{len(sod_violations)} 筆")
print(f"③ 超過內部deadline(10:30)：{len(late_internal)} 筆")
print(f"③ 超過法規deadline(11:00)：{len(late_regulatory)} 筆")
print(f"④ 利率波動異常(>10%)：{len(rate_anomalies)} 筆")
print(f"⑤ 漏報：{len(missing_df)} 筆")
print("="*50)

① 重複報送：4 筆
② SoD違規：27 筆
③ 超過內部deadline(10:30)：91 筆
③ 超過法規deadline(11:00)：64 筆
④ 利率波動異常(>10%)：149 筆
⑤ 漏報：0 筆


In [3]:
# 輸出異常報告到Excel
output_path = 'rate_submission_audit_report.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    duplicates.to_excel(writer, sheet_name='①重複報送', index=False)
    sod_violations.to_excel(writer, sheet_name='②SoD違規', index=False)
    late_internal.to_excel(writer, sheet_name='③超過內部Deadline', index=False)
    late_regulatory.to_excel(writer, sheet_name='③超過法規Deadline', index=False)
    rate_anomalies[['submission_date','tenor','rate','prev_rate','rate_change_pct']].to_excel(writer, sheet_name='④利率波動異常', index=False)
    missing_df.to_excel(writer, sheet_name='⑤漏報', index=False)

print(f"報告已輸出：{output_path}")

報告已輸出：rate_submission_audit_report.xlsx
